In [ ]:
# Submission path setup: run notebooks from any submission subfolder.
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'Functions.ipynb').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)


In [ ]:
# !pip install "numpy<2.0"
# !pip install torch==2.2.2 torchvision==0.17.2 monai
# !pip install import-ipynb
# !pip install nibabel 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import monai
import import_ipynb

from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    LoadImaged,
    NormalizeIntensity,
    ResizeWithPadOrCrop,
    Spacingd,
)

from Functions import patients_dicts, par_voxelsize


In [ ]:
patient_id = "patient001"
target_shape = (256, 256, 16)
finer_z_spacings = [5]

full_dict_list = patients_dicts("train")
sample_dict = next(sample for sample in full_dict_list if sample["ID"] == patient_id)

train_dataset_raw = monai.data.Dataset(
    full_dict_list,
    transform=Compose([
        LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
        EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
    ]),
)

mean_voxel, std_voxel, max_voxelsize = par_voxelsize([], train_dataset_raw)
current_z_spacing = float(mean_voxel[2])
all_z_spacings = [current_z_spacing] + finer_z_spacings

print("Median voxel size:", mean_voxel)
print("Comparing z spacings:", all_z_spacings)


In [ ]:
resize_to_model_input = ResizeWithPadOrCrop(spatial_size=target_shape)
normalize_image = NormalizeIntensity(nonzero=True, channel_wise=True)


def clone_volume(x):
    return x.clone() if hasattr(x, "clone") else np.array(x, copy=True)


def preprocess_with_z_spacing(sample_dict, z_spacing):
    transform = Compose([
        LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
        EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
        Spacingd(
            keys=["imgED", "maskED", "imgES", "maskES"],
            pixdim=(mean_voxel[0], mean_voxel[1], z_spacing),
            mode=("bilinear", "nearest", "bilinear", "nearest"),
            ensure_same_shape=True,
            align_corners=False,
        ),
    ])
    sample = transform(sample_dict)

    out = {"ID": sample["ID"], "Disease": sample["Disease"]}
    for phase in ["ED", "ES"]:
        image = clone_volume(sample[f"img{phase}"])
        mask = clone_volume(sample[f"mask{phase}"])
        out[f"raw_img{phase}"] = np.asarray(image).squeeze()
        out[f"raw_mask{phase}"] = np.asarray(mask).squeeze()
        out[f"img{phase}"] = np.asarray(normalize_image(resize_to_model_input(image))).squeeze()
        out[f"mask{phase}"] = np.asarray(resize_to_model_input(mask)).squeeze()
    return out


def occupied_indices(mask):
    return np.where((mask > 0).any(axis=(0, 1)))[0].tolist()


def plot_phase_comparison(phase, processed_samples, z_spacings):
    n_rows = len(z_spacings) * 2
    fig, axes = plt.subplots(n_rows, target_shape[-1], figsize=(2.0 * target_shape[-1], 2.0 * n_rows))

    if n_rows == 2:
        axes = np.expand_dims(axes, 0)

    for row_block, (z_spacing, sample) in enumerate(zip(z_spacings, processed_samples)):
        img = sample[f"img{phase}"]
        mask = sample[f"mask{phase}"]
        raw_occ = occupied_indices(sample[f"raw_mask{phase}"])
        final_occ = occupied_indices(mask)
        print(
            f"{phase} | z={z_spacing} | raw shape={sample[f'raw_img{phase}'].shape} | "
            f"raw occupied={raw_occ} | final occupied={final_occ}"
        )

        img_row = row_block * 2
        mask_row = img_row + 1
        for z in range(target_shape[-1]):
            axes[img_row, z].imshow(img[:, :, z], cmap="gray")
            axes[img_row, z].axis("off")
            axes[img_row, z].set_title(f"z={z}")

            axes[mask_row, z].imshow(mask[:, :, z], cmap="viridis", vmin=0, vmax=3)
            axes[mask_row, z].axis("off")

        label = f"z={z_spacing:g} img"
        if z_spacing == current_z_spacing:
            label = f"current z={z_spacing:g} img"
        axes[img_row, 0].set_ylabel(label)
        axes[mask_row, 0].set_ylabel(label.replace("img", "mask"))

    fig.suptitle(f"{processed_samples[0]['ID']} | {phase} | current vs finer z spacing", y=1.02)
    fig.tight_layout()
    plt.show()


processed_samples = [preprocess_with_z_spacing(sample_dict, z) for z in all_z_spacings]
print("Loaded:", processed_samples[0]["ID"], processed_samples[0]["Disease"])

plot_phase_comparison("ED", processed_samples, all_z_spacings)
plot_phase_comparison("ES", processed_samples, all_z_spacings)


In [ ]:
import torch

xy_crop_margin_ratio = 0.35
xy_crop_z_spacing = finer_z_spacings[0]


def crop_xy_from_mask(image, mask, margin_ratio=0.35):
    foreground = np.where(mask > 0)
    if len(foreground[0]) == 0:
        return image, mask, (0, image.shape[0] - 1, 0, image.shape[1] - 1)

    min_y, max_y = int(foreground[0].min()), int(foreground[0].max())
    min_x, max_x = int(foreground[1].min()), int(foreground[1].max())

    margin_y = int((max_y - min_y + 1) * margin_ratio)
    margin_x = int((max_x - min_x + 1) * margin_ratio)

    crop_min_y = max(min_y - margin_y, 0)
    crop_max_y = min(max_y + margin_y, image.shape[0] - 1)
    crop_min_x = max(min_x - margin_x, 0)
    crop_max_x = min(max_x + margin_x, image.shape[1] - 1)

    cropped_image = image[crop_min_y : crop_max_y + 1, crop_min_x : crop_max_x + 1, :]
    cropped_mask = mask[crop_min_y : crop_max_y + 1, crop_min_x : crop_max_x + 1, :]
    return cropped_image, cropped_mask, (crop_min_y, crop_max_y, crop_min_x, crop_max_x)


def resize_xy_only_to_model_input(image, mask):
    out_img = np.zeros(target_shape, dtype=np.float32)
    out_mask = np.zeros(target_shape, dtype=mask.dtype)

    for z in range(image.shape[-1]):
        image_slice = torch.as_tensor(image[:, :, z][None, None], dtype=torch.float32)
        mask_slice = torch.as_tensor(mask[:, :, z][None, None], dtype=torch.float32)

        out_img[:, :, z] = torch.nn.functional.interpolate(
            image_slice,
            size=target_shape[:2],
            mode="bilinear",
            align_corners=False,
        )[0, 0].cpu().numpy()

        out_mask[:, :, z] = np.rint(
            torch.nn.functional.interpolate(
                mask_slice,
                size=target_shape[:2],
                mode="nearest",
            )[0, 0].cpu().numpy()
        ).astype(mask.dtype)

    out_img = np.asarray(normalize_image(out_img[None])).squeeze()
    return out_img, out_mask


def prepare_three_stage_comparison(sample_dict, z_spacing, margin_ratio=0.35):
    current = preprocess_with_z_spacing(sample_dict, current_z_spacing)
    z_only = preprocess_with_z_spacing(sample_dict, z_spacing)

    out = {
        "ID": current["ID"],
        "Disease": current["Disease"],
        "z_spacing": z_spacing,
        "margin_ratio": margin_ratio,
    }

    for phase in ["ED", "ES"]:
        z_img = np.asarray(z_only[f"img{phase}"])
        z_mask = np.asarray(z_only[f"mask{phase}"])
        cropped_z_img, cropped_z_mask, bbox = crop_xy_from_mask(z_img, z_mask, margin_ratio)
        xyz_img, xyz_mask = resize_xy_only_to_model_input(cropped_z_img, cropped_z_mask)

        out[f"current_img{phase}"] = np.asarray(current[f"img{phase}"])
        out[f"current_mask{phase}"] = np.asarray(current[f"mask{phase}"])
        out[f"z_img{phase}"] = z_img
        out[f"z_mask{phase}"] = z_mask
        out[f"xyz_img{phase}"] = xyz_img
        out[f"xyz_mask{phase}"] = xyz_mask
        out[f"cropped_z_img{phase}"] = cropped_z_img
        out[f"cropped_z_mask{phase}"] = cropped_z_mask
        out[f"bbox{phase}"] = bbox

    return out


In [ ]:
def plot_three_stage_comparison(phase, comparison):
    fig, axes = plt.subplots(6, target_shape[-1], figsize=(2.0 * target_shape[-1], 12.0))

    current_img = comparison[f"current_img{phase}"]
    current_mask = comparison[f"current_mask{phase}"]
    z_img = comparison[f"z_img{phase}"]
    z_mask = comparison[f"z_mask{phase}"]
    xyz_img = comparison[f"xyz_img{phase}"]
    xyz_mask = comparison[f"xyz_mask{phase}"]
    bbox = comparison[f"bbox{phase}"]
    cropped_shape = comparison[f"cropped_z_img{phase}"].shape

    print(
        f"{phase} | z={comparison['z_spacing']} | "
        f"bbox(ymin,ymax,xmin,xmax)={bbox} | cropped z-only shape={cropped_shape}"
    )

    row_labels = [
        "current img",
        "current mask",
        "z-only img",
        "z-only mask",
        "z+xy img",
        "z+xy mask",
    ]
    row_arrays = [current_img, current_mask, z_img, z_mask, xyz_img, xyz_mask]
    row_cmaps = ["gray", "viridis", "gray", "viridis", "gray", "viridis"]

    for row_idx, (row_array, cmap) in enumerate(zip(row_arrays, row_cmaps)):
        for z in range(target_shape[-1]):
            if cmap == "viridis":
                axes[row_idx, z].imshow(row_array[:, :, z], cmap=cmap, vmin=0, vmax=3)
            else:
                axes[row_idx, z].imshow(row_array[:, :, z], cmap=cmap)
            axes[row_idx, z].axis("off")
            if row_idx == 0:
                axes[row_idx, z].set_title(f"z={z}")
        axes[row_idx, 0].set_ylabel(row_labels[row_idx])

    fig.suptitle(
        f"{comparison['ID']} | {phase} | current -> z-only -> z+xy",
        y=1.02,
    )
    fig.tight_layout()
    plt.show()


three_stage_comparison = prepare_three_stage_comparison(
    sample_dict,
    z_spacing=xy_crop_z_spacing,
    margin_ratio=xy_crop_margin_ratio,
)

print("Loaded:", three_stage_comparison["ID"], three_stage_comparison["Disease"])
print("Comparing current input, z-only, and z+xy crop at z spacing:", xy_crop_z_spacing)

plot_three_stage_comparison("ED", three_stage_comparison)
plot_three_stage_comparison("ES", three_stage_comparison)
